# Clase 15 — Casos de Estudio: Operaciones clave-valor e Introducción a DataFrames

Este notebook desarrolla **únicamente los casos de estudio, sus tareas y preguntas** de:

- **Sesión 1: Pair RDDs y operaciones clave-valor** → 🎵 *Caso de Estudio 1: MusicStream Analytics*
- **Sesión 2: Spark DataFrames I — Introducción** → 🌾 *Caso de Estudio 2: AgroData Cooperativa*

Entorno: Apache Spark 4.1.1 + Scala 2.13 (Almond kernel) en modo `local[*]`.

## 0. Inicialización de Spark

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Clase15-Casos-PairRDD-DataFrames")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._
val sc = spark.sparkContext

println(s"✅ Spark ${spark.version} iniciado en ${sc.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/30 07:22:36 INFO SparkContext: Running Spark version 4.1.1
26/04/30 07:22:36 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/30 07:22:36 INFO SparkContext: Java version 17.0.18+8
26/04/30 07:22:37 INFO ResourceUtils: ==============================================================
26/04/30 07:22:37 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/30 07:22:37 INFO ResourceUtils: ==============================================================
26/04/30 07:22:37 INFO SparkContext: Submitted application: Clase15-Casos-PairRDD-DataFrames
26/04/30 07:22:37 INFO SecurityManager: Changing view acls to: gre
26/04/30 07:22:37 INFO SecurityManager: Changing modify acls to: gre
26/04/30 07:22:37 INFO SecurityManager: Changing view acls groups to: gre
26/04/30 07:22:37 INFO SecurityManager: Changing modify acls groups to: gre
26/04/30 07:22:37 INFO SecurityManager: SecurityManager: 

✅ Spark 4.1.1 iniciado en local[*]


import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@57eb8565
import spark.implicits._
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@78a70b9e

---

# 🎵 Caso de Estudio 1 — MusicStream Analytics

**Sesión 1: Pair RDDs y operaciones clave-valor**

Plataforma de streaming musical en España, México, Argentina y Colombia. Analizamos un log del fin de semana para extraer métricas de negocio.

**Formato:** `usuario_id,pais,genero,cancion_id,duracion_segundos,completada`

## 📦 Datos del caso

In [2]:
val logReproducciones = sc.parallelize(List(
  "u001,España,Pop,Rosalía|Malamente,210,S",
  "u002,México,Reggaeton,BadBunny|Tití,185,S",
  "u001,España,Rock,Vetusta|Cable,240,S",
  "u003,Argentina,Jazz,MilesDavis|So What,320,S",
  "u004,Colombia,Reggaeton,BadBunny|Tití,185,N",
  "u002,México,Pop,Rosalía|Malamente,210,S",
  "u005,España,Electronica,Daft Punk|Harder,224,S",
  "u003,Argentina,Pop,Shakira|Waka,198,N",
  "u006,Colombia,Jazz,ColtraneJ|Giant Steps,280,S",
  "u001,España,Electronica,Daft Punk|Harder,224,N",
  "u007,México,Rock,Mägo|Oro,195,S",
  "u004,Colombia,Pop,Shakira|Waka,198,S",
  "u002,México,Jazz,MilesDavis|So What,320,S",
  "u005,España,Reggaeton,BadBunny|Tití,185,S",
  "u008,Argentina,Rock,Soda|De Música,212,S",
  "u006,Colombia,Electronica,Daft Punk|Harder,224,N",
  "u003,Argentina,Reggaeton,BadBunny|Tití,185,S",
  "u007,México,Pop,Rosalía|Malamente,210,N",
  "u009,España,Jazz,ColtraneJ|Giant Steps,280,S",
  "u008,Argentina,Pop,Rosalía|Malamente,210,S",
  "u010,Colombia,Rock,Mägo|Oro,195,S",
  "u001,España,Reggaeton,J Balvin|Mi Gente,178,S",
  "u009,España,Pop,Shakira|Waka,198,S",
  "u005,España,Rock,Vetusta|Cable,240,S",
  "u010,Colombia,Reggaeton,J Balvin|Mi Gente,178,S",
  "u006,Colombia,Pop,Rosalía|Malamente,210,S",
  "u007,México,Electronica,Daft Punk|Harder,224,S",
  "u002,México,Rock,Soda|De Música,212,S",
  "u008,Argentina,Jazz,MilesDavis|So What,320,N",
  "u004,Colombia,Electronica,Daft Punk|Harder,224,S"
))

logReproducciones.cache()
println(s"Reproducciones cargadas: ${logReproducciones.count()}")

Reproducciones cargadas: 30


logReproducciones: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:1
res2_1: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:1

## ❓ Pregunta 1 — Minutos totales escuchados por país

**Operación principal:** `reduceByKey`.

In [3]:
val segPorPais = logReproducciones
  .map { linea =>
    val c = linea.split(",")
    (c(1), c(4).toInt)        // (pais, segundos)
  }
  .reduceByKey(_ + _)
  .sortBy(-_._2)

println("=== Minutos escuchados por país ===")
segPorPais.collect().foreach { case (pais, seg) =>
  println(f"  ${pais}%-10s → ${seg / 60.0}%.1f min")
}

=== Minutos escuchados por país ===
  España     → 33,0 min
  Colombia   → 28,2 min
  México     → 25,9 min
  Argentina  → 24,1 min


segPorPais: org.apache.spark.rdd.RDD[(String, Int)] = MapPartitionsRDD[7] at sortBy at cmd3.sc:7

## ❓ Pregunta 2 — Tasa de canciones completadas por género

**Operación principal:** `aggregateByKey` con acumulador `(completadas, total)`.

In [4]:
val tasaPorGenero = logReproducciones
  .map { linea =>
    val c = linea.split(",")
    (c(2), c(5))               // (genero, completada S/N)
  }
  .aggregateByKey((0, 0))(
    (acc, comp) => (acc._1 + (if (comp == "S") 1 else 0), acc._2 + 1),
    (a, b)     => (a._1 + b._1, a._2 + b._2)
  )

println("=== Tasa de compleción por género ===")
tasaPorGenero.collect().sortBy { case (_, (c, t)) => -c.toDouble / t }.foreach {
  case (genero, (completadas, total)) =>
    val tasa = completadas.toDouble / total * 100
    println(f"  ${genero}%-12s → ${tasa}%5.1f%%  ($completadas/$total reproducciones)")
}

=== Tasa de compleción por género ===
  Rock         → 100,0%  (6/6 reproducciones)
  Reggaeton    →  83,3%  (5/6 reproducciones)
  Jazz         →  80,0%  (4/5 reproducciones)
  Pop          →  75,0%  (6/8 reproducciones)
  Electronica  →  60,0%  (3/5 reproducciones)


tasaPorGenero: org.apache.spark.rdd.RDD[(String, (Int, Int))] = ShuffledRDD[9] at aggregateByKey at cmd4.sc:6

## ❓ Pregunta 3 — Géneros que escucha cada usuario

**Operación principal:** `combineByKey` acumulando una lista sin duplicados.

In [5]:
val generosPorUsuario = logReproducciones
  .map { linea =>
    val c = linea.split(",")
    (c(0), c(2))                // (usuario, genero)
  }
  .combineByKey(
    (g: String) => List(g),
    (acc: List[String], v: String) => if (acc.contains(v)) acc else acc :+ v,
    (a: List[String], b: List[String]) => (a ++ b).distinct
  )
  .mapValues(_.sorted)
  .sortByKey()

println("=== Géneros explorados por usuario ===")
generosPorUsuario.collect().foreach { case (u, gs) =>
  println(s"  $u → ${gs.mkString(", ")}")
}

=== Géneros explorados por usuario ===
  u001 → Electronica, Pop, Reggaeton, Rock
  u002 → Jazz, Pop, Reggaeton, Rock
  u003 → Jazz, Pop, Reggaeton
  u004 → Electronica, Pop, Reggaeton
  u005 → Electronica, Reggaeton, Rock
  u006 → Electronica, Jazz, Pop
  u007 → Electronica, Pop, Rock
  u008 → Jazz, Pop, Rock
  u009 → Jazz, Pop
  u010 → Reggaeton, Rock


generosPorUsuario: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[15] at sortByKey at cmd5.sc:12

## ❓ Pregunta 4 — Estadísticas de escucha por país: total, media y máximo

**Operación principal:** `aggregateByKey` con acumulador triple `(suma, max, count)`.

In [6]:
val statsPorPais = logReproducciones
  .map { linea =>
    val c = linea.split(",")
    (c(1), c(4).toInt)          // (pais, segundos)
  }
  .aggregateByKey((0, 0, 0))(
    (acc, seg) => (acc._1 + seg, math.max(acc._2, seg), acc._3 + 1),
    (a, b)     => (a._1 + b._1, math.max(a._2, b._2), a._3 + b._3)
  )
  .sortByKey()

println("=== Resumen de escucha por país ===")
println(f"${"País"}%-12s ${"Total"}%-10s ${"Media"}%-10s ${"Máximo"}%-10s ${"Reproduc."}%-10s")
println("-" * 56)
statsPorPais.collect().foreach { case (pais, (suma, maxv, n)) =>
  val media = suma.toDouble / n
  println(f"${pais}%-12s ${suma}%-4d s    ${media}%6.1f s   ${maxv}%-4d s    $n")
}

=== Resumen de escucha por país ===
País         Total      Media      Máximo     Reproduc. 
--------------------------------------------------------
Argentina    1445 s     240,8 s   320  s    6
Colombia     1694 s     211,8 s   280  s    8
España       1979 s     219,9 s   280  s    9
México       1556 s     222,3 s   320  s    7


statsPorPais: org.apache.spark.rdd.RDD[(String, (Int, Int, Int))] = ShuffledRDD[20] at sortByKey at cmd6.sc:10

## ❓ Pregunta 5 — Canciones distintas reproducidas por género

**Operación principal:** `combineByKey` con `Set[String]` + `mapValues(_.size)`.

In [7]:
val cancionesUnicasPorGenero = logReproducciones
  .map { linea =>
    val c = linea.split(",")
    (c(2), c(3))                 // (genero, cancion_id)
  }
  .combineByKey(
    (c: String) => Set(c),
    (acc: Set[String], c: String) => acc + c,
    (a: Set[String], b: Set[String]) => a ++ b
  )
  .mapValues(_.size)
  .sortByKey()

println("=== Canciones únicas por género ===")
cancionesUnicasPorGenero.collect().foreach { case (g, n) =>
  val txt = if (n == 1) "canción única" else "canciones únicas"
  println(f"  ${g}%-12s → $n $txt")
}

=== Canciones únicas por género ===
  Electronica  → 1 canción única
  Jazz         → 2 canciones únicas
  Pop          → 2 canciones únicas
  Reggaeton    → 2 canciones únicas
  Rock         → 3 canciones únicas


cancionesUnicasPorGenero: org.apache.spark.rdd.RDD[(String, Int)] = ShuffledRDD[26] at sortByKey at cmd7.sc:12

## ❓ Pregunta 6 — Guía de operaciones (¿por qué no `groupByKey`?)

| Pregunta | Operación principal | ¿Por qué esa y no `groupByKey`? |
| --- | --- | --- |
| 1 — Minutos por país | `reduceByKey` | Solo necesitamos un agregado (suma). `reduceByKey` **precombina localmente** en cada partición antes del shuffle, moviendo mucha menos información por la red que `groupByKey`, que enviaría todos los segundos sin agregar. |
| 2 — Tasa de compleción | `aggregateByKey` | El acumulador `(completadas, total)` tiene un **tipo distinto** al valor de entrada (`String` "S"/"N"). `reduceByKey` no permite cambiar el tipo. `groupByKey` recolectaría todos los "S"/"N" sin agregar, gastando mucha memoria. |
| 3 — Géneros por usuario | `combineByKey` | Necesitamos construir una **colección única** (lista sin duplicados). `combineByKey` permite definir explícitamente cómo crear el acumulador, cómo añadir un elemento y cómo fusionar dos acumuladores, evitando duplicados durante todo el proceso. `groupByKey` daría una lista con repeticiones que habría que limpiar después. |
| 4 — Estadísticas por país | `aggregateByKey` | Tres métricas simultáneas `(suma, max, count)` requieren un **acumulador estructurado** distinto al valor de entrada. Una sola pasada en lugar de tres acciones independientes; muy superior a `groupByKey`, que materializaría toda la lista de segundos en memoria. |
| 5 — Canciones únicas | `combineByKey` + `mapValues` | Acumulamos un `Set[String]` que **deduplica en el origen**. Con `groupByKey` cargaríamos toda la lista de canciones (con repetidos) en el reducer y luego haría falta deduplicar; con `combineByKey` la deduplicación ocurre incluso antes del shuffle. |

> 💡 **Regla práctica:** `groupByKey` es prácticamente siempre evitable. Solo lo usaríamos si necesitásemos la lista completa de valores tal cual y no nos sirviera ningún acumulador.

In [8]:
logReproducciones.unpersist()
println("✅ Caché de logReproducciones liberada.")

✅ Caché de logReproducciones liberada.


res8_0: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:1

---

# 🌾 Caso de Estudio 2 — AgroData Cooperativa

**Sesión 2: Spark DataFrames I — Introducción**

Cooperativa agrícola con productores en cinco provincias españolas. Centralizamos los datos exportados en CSV (parcelas) y JSON (catálogo de productos) para extraer un primer informe.

## 🛠️ Generación de los ficheros de datos

En lugar de crearlos manualmente, los generamos desde Scala en `C:\Curso-Scala\datos\agrodata\`.

In [9]:
import java.nio.file.{Files, Paths, StandardOpenOption}
import java.nio.charset.StandardCharsets

val rutaBase = Paths.get("C:/Curso-Scala/datos/agrodata")
Files.createDirectories(rutaBase)

val parcelasCsv =
  """id_parcela,provincia,municipio,superficie_ha,cultivo,año_alta,en_produccion,rendimiento_kg_ha
    |P001,Sevilla,Carmona,12.5,Naranja,2015,true,28000
    |P002,Huelva,Lepe,8.3,Fresa,2018,true,45000
    |P003,Almería,Níjar,25.0,Tomate,2012,true,85000
    |P004,Sevilla,Écija,6.7,Aceituna,2009,false,3200
    |P005,Murcia,Totana,15.2,Limón,2016,true,22000
    |P006,Almería,El Ejido,30.1,Pimiento,2014,true,62000
    |P007,Huelva,Moguer,9.8,Fresa,2020,true,41000
    |P008,Murcia,Lorca,18.4,Melocotón,2011,false,9500
    |P009,Sevilla,Utrera,22.0,Naranja,2013,true,31000
    |P010,Almería,Vícar,11.6,Pepino,2019,true,74000
    |P011,Murcia,Alhama,7.9,Limón,2017,true,20500
    |P012,Huelva,Cartaya,14.3,Fresa,2015,true,43000
    |P013,Sevilla,Marchena,5.2,Aceituna,2010,false,2900
    |P014,Almería,Roquetas,28.7,Tomate,2011,true,88000
    |P015,Murcia,Mazarrón,16.5,Pimiento,2018,true,58000
    |""".stripMargin

val productosJson =
  """[
    |  {"codigo":"NAR","nombre":"Naranja","familia":"Citrico","precio_mercado_euro_kg":0.45,"demanda_exportacion":"Alta","certificacion_eco":false},
    |  {"codigo":"FRE","nombre":"Fresa","familia":"Baya","precio_mercado_euro_kg":2.10,"demanda_exportacion":"Muy Alta","certificacion_eco":true},
    |  {"codigo":"TOM","nombre":"Tomate","familia":"Hortaliza","precio_mercado_euro_kg":0.85,"demanda_exportacion":"Alta","certificacion_eco":false},
    |  {"codigo":"ACE","nombre":"Aceituna","familia":"Oleaginosa","precio_mercado_euro_kg":0.60,"demanda_exportacion":"Media","certificacion_eco":true},
    |  {"codigo":"LIM","nombre":"Limón","familia":"Citrico","precio_mercado_euro_kg":0.55,"demanda_exportacion":"Alta","certificacion_eco":false},
    |  {"codigo":"PIM","nombre":"Pimiento","familia":"Hortaliza","precio_mercado_euro_kg":1.20,"demanda_exportacion":"Muy Alta","certificacion_eco":true},
    |  {"codigo":"PEP","nombre":"Pepino","familia":"Hortaliza","precio_mercado_euro_kg":0.70,"demanda_exportacion":"Media","certificacion_eco":false},
    |  {"codigo":"MEL","nombre":"Melocotón","familia":"Drupa","precio_mercado_euro_kg":1.35,"demanda_exportacion":"Media","certificacion_eco":false}
    |]
    |""".stripMargin

Files.write(rutaBase.resolve("parcelas.csv"), parcelasCsv.getBytes(StandardCharsets.UTF_8))
Files.write(rutaBase.resolve("productos.json"), productosJson.getBytes(StandardCharsets.UTF_8))

println(s"✅ Ficheros creados en: ${rutaBase.toAbsolutePath}")

✅ Ficheros creados en: C:\Curso-Scala\datos\agrodata


import java.nio.file.{Files, Paths, StandardOpenOption}
import java.nio.charset.StandardCharsets
rutaBase: java.nio.file.Path = C:\Curso-Scala\datos\agrodata
res9_3: java.nio.file.Path = C:\Curso-Scala\datos\agrodata
parcelasCsv: String = """id_parcela,provincia,municipio,superficie_ha,cultivo,año_alta,en_produccion,rendimiento_kg_ha
P001,Sevilla,Carmona,12.5,Naranja,2015,true,28000
P002,Huelva,Lepe,8.3,Fresa,2018,true,45000
P003,Almería,Níjar,25.0,Tomate,2012,true,85000
P004,Sevilla,Écija,6.7,Aceituna,2009,false,3200
P005,Murcia,Totana,15.2,Limón,2016,true,22000
P006,Almería,El Ejido,30.1,Pimiento,2014,true,62000
P007,Huelva,Moguer,9.8,Fresa,2020,true,41000
P008,Murcia,Lorca,18.4,Melocotón,2011,false,9500
P009,Sevilla,Utrera,22.0,Naranja,2013,true,31000
P010,Almería,Vícar,11.6,Pepino,2019,true,74000
P011,Murcia,Alhama,7.9,Limón,2017,true,20500
P012,Huelva,Cartaya,14.3,Fresa,2015,true,43000
P013,Sevilla,Marchena,5.2,Aceituna,2010,false,2900
P014,Almería,Roquetas,28.7,Tomate,2011,true,8

## 📋 Tarea 1 — Inicialización del entorno

La `SparkSession` ya se creó en la celda 0 con `appName = "Clase15-Casos-PairRDD-DataFrames"`. Aquí confirmamos que todo está listo y dejamos preparados los imports adicionales para esta sesión.

In [10]:
import org.apache.spark.sql.types._
// spark.implicits._ ya importado en la celda 0

println(s"✅ AgroData Analytics iniciado — Spark ${spark.version}")

✅ AgroData Analytics iniciado — Spark 4.1.1


import org.apache.spark.sql.types._

## 📋 Tarea 2 — Carga del CSV de parcelas con `inferSchema`

In [11]:
val dfParcelasInf = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Curso-Scala/datos/agrodata/parcelas.csv")

println("=== Primeras 5 parcelas ===")
dfParcelasInf.show(5)

println("=== Schema inferido ===")
dfParcelasInf.printSchema()

println(s"Total de parcelas registradas: ${dfParcelasInf.count()}")
println(s"Columnas: ${dfParcelasInf.columns.mkString(" | ")}")

=== Primeras 5 parcelas ===
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|id_parcela|provincia|municipio|superficie_ha| cultivo|año_alta|en_produccion|rendimiento_kg_ha|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|      P001|  Sevilla|  Carmona|         12.5| Naranja|    2015|         true|            28000|
|      P002|   Huelva|     Lepe|          8.3|   Fresa|    2018|         true|            45000|
|      P003|  Almería|    Níjar|         25.0|  Tomate|    2012|         true|            85000|
|      P004|  Sevilla|    Écija|          6.7|Aceituna|    2009|        false|             3200|
|      P005|   Murcia|   Totana|         15.2|   Limón|    2016|         true|            22000|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
only showing top 5 rows
=== Schema inferido ===
root
 |-- id_parcela: string (nullable = true)
 |--

dfParcelasInf: org.apache.spark.sql.package.DataFrame = [id_parcela: string, provincia: string ... 6 more fields]

## 📋 Tarea 3 — Definir el schema manualmente y recargar

Correcciones requeridas:
- `id_parcela`: `nullable = false` (clave que nunca debe ser nula)
- `rendimiento_kg_ha`: `DoubleType` en lugar de `IntegerType` para permitir cálculos con decimales

In [12]:
val schemaParcelas = StructType(Array(
  StructField("id_parcela",        StringType,  nullable = false),
  StructField("provincia",         StringType,  nullable = true),
  StructField("municipio",         StringType,  nullable = true),
  StructField("superficie_ha",     DoubleType,  nullable = true),
  StructField("cultivo",           StringType,  nullable = true),
  StructField("año_alta",          IntegerType, nullable = true),
  StructField("en_produccion",     BooleanType, nullable = true),
  StructField("rendimiento_kg_ha", DoubleType,  nullable = true)
))

val dfParcelas = spark.read
  .option("header", "true")
  .schema(schemaParcelas)
  .csv("C:/Curso-Scala/datos/agrodata/parcelas.csv")

println("=== Schema manual aplicado ===")
dfParcelas.printSchema()

println("=== Tipos por columna ===")
dfParcelas.dtypes.foreach { case (col, tipo) =>
  println(f"  ${col}%-18s → $tipo")
}

=== Schema manual aplicado ===
root
 |-- id_parcela: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- superficie_ha: double (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- año_alta: integer (nullable = true)
 |-- en_produccion: boolean (nullable = true)
 |-- rendimiento_kg_ha: double (nullable = true)

=== Tipos por columna ===
  id_parcela         → StringType
  provincia          → StringType
  municipio          → StringType
  superficie_ha      → DoubleType
  cultivo            → StringType
  año_alta           → IntegerType
  en_produccion      → BooleanType
  rendimiento_kg_ha  → DoubleType


schemaParcelas: StructType = Seq(
  StructField(
    name = "id_parcela",
    dataType = StringType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "provincia",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "municipio",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "superficie_ha",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "cultivo",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "año_alta",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "en_produccion",
...
dfParcelas: org.apache.spark.sql.package.DataFrame = [id_parcela: string, provincia: string ... 6 more fields]

## 📋 Tarea 4 — Primer informe estadístico de las parcelas

Usamos `describe()` para obtener resumen numérico (count, mean, stddev, min, max).

In [13]:
println("=== Estadísticas de superficie y rendimiento ===")
dfParcelas.describe("superficie_ha", "rendimiento_kg_ha").show()

println("=== Rango de años de alta ===")
dfParcelas.describe("año_alta").show()

=== Estadísticas de superficie y rendimiento ===
+-------+------------------+------------------+
|summary|     superficie_ha| rendimiento_kg_ha|
+-------+------------------+------------------+
|  count|                15|                15|
|   mean|15.479999999999999|40873.333333333336|
| stddev| 7.931240220077275|27911.479120008433|
|    min|               5.2|            2900.0|
|    max|              30.1|           88000.0|
+-------+------------------+------------------+

=== Rango de años de alta ===
+-------+------------------+
|summary|          año_alta|
+-------+------------------+
|  count|                15|
|   mean|2014.5333333333333|
| stddev|3.4613512362879963|
|    min|              2009|
|    max|              2020|
+-------+------------------+



## 📋 Tarea 5 — Carga del catálogo de productos en JSON

In [14]:
val dfProductosInf = spark.read
  .option("multiline", "true")
  .json("C:/Curso-Scala/datos/agrodata/productos.json")

println("=== Catálogo de productos ===")
dfProductosInf.show(truncate = false)

println("=== Schema del JSON ===")
dfProductosInf.printSchema()

// Anotación:
//   - precio_mercado_euro_kg → DoubleType (correcto, son precios decimales)
//   - certificacion_eco      → BooleanType (correcto, true/false en el JSON)
// Spark ordena las columnas alfabéticamente al inferir desde JSON.
println("📌 precio_mercado_euro_kg → DoubleType ✅")
println("📌 certificacion_eco      → BooleanType ✅")
println("📌 (Las columnas aparecen en orden alfabético, comportamiento normal del lector JSON)")

=== Catálogo de productos ===
+-----------------+------+-------------------+----------+---------+----------------------+
|certificacion_eco|codigo|demanda_exportacion|familia   |nombre   |precio_mercado_euro_kg|
+-----------------+------+-------------------+----------+---------+----------------------+
|false            |NAR   |Alta               |Citrico   |Naranja  |0.45                  |
|true             |FRE   |Muy Alta           |Baya      |Fresa    |2.1                   |
|false            |TOM   |Alta               |Hortaliza |Tomate   |0.85                  |
|true             |ACE   |Media              |Oleaginosa|Aceituna |0.6                   |
|false            |LIM   |Alta               |Citrico   |Limón    |0.55                  |
|true             |PIM   |Muy Alta           |Hortaliza |Pimiento |1.2                   |
|false            |PEP   |Media              |Hortaliza |Pepino   |0.7                   |
|false            |MEL   |Media              |Drupa     |Mel

dfProductosInf: org.apache.spark.sql.package.DataFrame = [certificacion_eco: boolean, codigo: string ... 4 more fields]

## 📋 Tarea 6 — Schema manual para el catálogo de productos

Restricciones:
- `codigo`: `nullable = false` (clave del producto)
- `precio_mercado_euro_kg`: `DoubleType` con `nullable = false`

In [15]:
val schemaProductos = StructType(Array(
  StructField("codigo",                 StringType,  nullable = false),
  StructField("nombre",                 StringType,  nullable = true),
  StructField("familia",                StringType,  nullable = true),
  StructField("precio_mercado_euro_kg", DoubleType,  nullable = false),
  StructField("demanda_exportacion",    StringType,  nullable = true),
  StructField("certificacion_eco",      BooleanType, nullable = true)
))

val dfProductos = spark.read
  .option("multiline", "true")
  .schema(schemaProductos)
  .json("C:/Curso-Scala/datos/agrodata/productos.json")

println("=== Schema manual del catálogo ===")
dfProductos.printSchema()

println("=== Comparativa de tipos: inferido vs manual ===")
val tiposInf    = dfProductosInf.dtypes.toMap
val tiposManual = dfProductos.dtypes.toMap
val todasLasCols = (tiposInf.keySet ++ tiposManual.keySet).toSeq.sorted
println(f"  ${"Columna"}%-25s ${"Inferido"}%-15s ${"Manual"}%-15s")
println("  " + "-" * 55)
todasLasCols.foreach { c =>
  val a = tiposInf.getOrElse(c, "-")
  val b = tiposManual.getOrElse(c, "-")
  println(f"  ${c}%-25s ${a}%-15s ${b}%-15s")
}

=== Schema manual del catálogo ===
root
 |-- codigo: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- familia: string (nullable = true)
 |-- precio_mercado_euro_kg: double (nullable = true)
 |-- demanda_exportacion: string (nullable = true)
 |-- certificacion_eco: boolean (nullable = true)

=== Comparativa de tipos: inferido vs manual ===
  Columna                   Inferido        Manual         
  -------------------------------------------------------
  certificacion_eco         BooleanType     BooleanType    
  codigo                    StringType      StringType     
  demanda_exportacion       StringType      StringType     
  familia                   StringType      StringType     
  nombre                    StringType      StringType     
  precio_mercado_euro_kg    DoubleType      DoubleType     


schemaProductos: StructType = Seq(
  StructField(
    name = "codigo",
    dataType = StringType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "nombre",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "familia",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "precio_mercado_euro_kg",
    dataType = DoubleType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "demanda_exportacion",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "certificacion_eco",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfProductos: org.apache.spark.sql.package.DataFrame = [codigo: string, nombre: string ... 4 more fields]
tiposInf: Map[String, String] = HashMap(
  "codigo" -> "StringType",
  "certificacion_eco" -> "BooleanType",
  "familia" -> "StringType",
  "precio_mercado_euro_kg" -> "Doub

## 📋 Tarea 7 — DataFrame de resumen desde colección en memoria

Creamos a mano una tabla resumen por provincia con `Seq(...).toDF(...)`.

In [16]:
val resumenProvincias = Seq(
  ("Almería", 4, 95.4, 4),
  ("Huelva",  3, 32.4, 3),
  ("Murcia",  4, 58.0, 3),
  ("Sevilla", 4, 46.4, 2)
).toDF("provincia", "num_parcelas", "superficie_total_ha", "parcelas_activas")

println("=== Resumen por provincia ===")
resumenProvincias.show()

println("=== Schema del resumen ===")
resumenProvincias.printSchema()

println(s"Filas:    ${resumenProvincias.count()}")
println(s"Columnas: ${resumenProvincias.columns.mkString(" | ")}")

=== Resumen por provincia ===
+---------+------------+-------------------+----------------+
|provincia|num_parcelas|superficie_total_ha|parcelas_activas|
+---------+------------+-------------------+----------------+
|  Almería|           4|               95.4|               4|
|   Huelva|           3|               32.4|               3|
|   Murcia|           4|               58.0|               3|
|  Sevilla|           4|               46.4|               2|
+---------+------------+-------------------+----------------+

=== Schema del resumen ===
root
 |-- provincia: string (nullable = true)
 |-- num_parcelas: integer (nullable = false)
 |-- superficie_total_ha: double (nullable = false)
 |-- parcelas_activas: integer (nullable = false)

Filas:    4
Columnas: provincia | num_parcelas | superficie_total_ha | parcelas_activas


resumenProvincias: org.apache.spark.sql.package.DataFrame = [provincia: string, num_parcelas: int ... 2 more fields]

In [ ]:
// spark.stop()